# Drone Imagery Search System — v2

This notebook extends the v1 prototype with the features called out in the
project proposal:

1. **EXIF/geotag extraction from JPG files** — when a folder has no `.MRK`
   or `.SRT` flight log, we now infer the flight path from geotags
   embedded in the photos themselves. The proposal lists this as a core
   need.
2. **Media inventory + flight detail page** — every flight now has a page
   that lists its photos and videos, with thumbnails. Search results link
   directly to it.
3. **Unified, date-filterable map** — `/api/flights` returns *both*
   professional and personal flights and honors the date range from the
   search form. Previously the map ignored the filter and only showed
   professional flights.
4. **Drone-identifier substring filter** — search by partial name in
   addition to date range and type.
5. **Local-mode synthetic data generator** — section 7 builds a fake folder
   tree with geotagged JPGs and a fake `.MRK`/`.SRT`, so you can test the
   pipeline end-to-end without mounting Google Drive.

Run sections 1–6 in Google Colab (or locally) against your real Drive
folder, or run section 7 first to play with the system using synthetic
data.

## 1. Setup

Install dependencies and detect whether we're running in Colab or
locally. The Colab-specific pieces (`google.colab.drive`, the proxy URL
trick) are only invoked when we're actually in Colab.

In [ ]:
# One-time install. piexif is the new dep — used for EXIF GPS extraction.
!pip install -q flask pandas pillow piexif

In [ ]:
import os
import re
import sqlite3
import threading
import time
import socket
import mimetypes
from datetime import datetime
from urllib.parse import quote

import pandas as pd
from PIL import Image
import piexif
from flask import (
    Flask, request, jsonify, render_template_string, send_file, abort,
)
from IPython.display import HTML, display

# Detect environment so the same notebook works in Colab and locally.
try:
    import google.colab            # noqa: F401
    IN_COLAB = True
    from google.colab import drive, output
except ImportError:
    IN_COLAB = False

DB_PATH = "drone_folders.db"
print(f"Environment: {'Google Colab' if IN_COLAB else 'local'}")
print(f"Database file: {os.path.abspath(DB_PATH)}")

In [ ]:
def setup_database(db_path=DB_PATH):
    """Create the v2 schema. Backwards-compatible with v1 tables; adds
    flight_media (per-file inventory) and indexes."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # --- Flight tables (unchanged from v1) ---
    cur.execute("""
        CREATE TABLE IF NOT EXISTS professional (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            date TEXT,
            folder_path TEXT
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS personal (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            date TEXT,
            folder_path TEXT
        )
    """)

    # --- Coordinate tables (unchanged from v1) ---
    cur.execute("""
        CREATE TABLE IF NOT EXISTS professional_coordinates (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            professional_name TEXT,
            lat REAL,
            lon REAL,
            sequence INTEGER,
            FOREIGN KEY(professional_name) REFERENCES professional(name)
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS personal_coordinates (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            personal_name TEXT,
            lat REAL,
            lon REAL,
            sequence INTEGER,
            FOREIGN KEY(personal_name) REFERENCES personal(name)
        )
    """)

    # --- NEW: per-file inventory so we can render a media gallery ---
    # flight_type is 'professional' or 'personal'. media_kind is one of
    # 'photo', 'video', 'log' (.mrk/.srt), 'other'.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS flight_media (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            flight_type TEXT NOT NULL,
            flight_name TEXT NOT NULL,
            filename TEXT NOT NULL,
            file_path TEXT NOT NULL,
            media_kind TEXT NOT NULL,
            lat REAL,
            lon REAL,
            taken_at TEXT,
            UNIQUE(flight_type, flight_name, filename)
        )
    """)

    # --- Indexes — these matter once the table grows past a few hundred rows ---
    cur.execute("CREATE INDEX IF NOT EXISTS idx_pro_date ON professional(date)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_per_date ON personal(date)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_pro_coords_name ON professional_coordinates(professional_name)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_per_coords_name ON personal_coordinates(personal_name)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_media_flight ON flight_media(flight_type, flight_name)")

    conn.commit()
    conn.close()
    print("Database schema ready.")

setup_database()

## 2. Coordinate extraction

Three extractors, one per source format:

* **`.MRK`** — DJI's RTK marker file (tab/comma-delimited rows, one per
  triggered photo). Used by professional flights.
* **`.SRT`** — DJI's video subtitle log, with `[latitude: x] [longitude: y]`
  on each frame line. Used by personal flights with video.
* **JPG EXIF** *(new in v2)* — for folders that have neither `.MRK` nor
  `.SRT`, we read GPS tags directly from the photo headers. This is the
  geotag fallback the proposal called out.

In [ ]:
def extract_lat_lon_mrk(mrk_file):
    """DJI .MRK file — coordinates appear as '<value>,Lat,<value>,Lon,...'."""
    coords = []
    try:
        with open(mrk_file, "r") as f:
            for idx, line in enumerate(f):
                parts = line.replace("\t", ",").split(",")
                parts = [p.strip() for p in parts if p.strip()]
                try:
                    lat = float(parts[parts.index("Lat") - 1])
                    lon = float(parts[parts.index("Lon") - 1])
                    coords.append((lat, lon, idx))
                except (ValueError, IndexError):
                    continue
    except Exception as e:
        print(f"Error reading {mrk_file}: {e}")
    return coords

In [ ]:
def extract_lat_lon_srt(srt_file):
    """DJI .SRT video log — frame lines contain '[latitude: x] [longitude: y]',
    sometimes inside a <font> wrapper. Match the bracketed tokens anywhere
    in the line rather than requiring the line to start with '['."""
    coords = []
    pat_lat = re.compile(r"\[latitude\s*:\s*(-?\d+\.\d+)\s*\]")
    pat_lon = re.compile(r"\[longitude\s*:\s*(-?\d+\.\d+)\s*\]")
    try:
        with open(srt_file, "r") as f:
            seq = 0
            for line in f:
                m_lat = pat_lat.search(line)
                m_lon = pat_lon.search(line)
                if m_lat and m_lon:
                    try:
                        coords.append((float(m_lat.group(1)),
                                       float(m_lon.group(1)), seq))
                        seq += 1
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error reading {srt_file}: {e}")
    return coords

In [ ]:
# --- NEW in v2: GPS extraction from JPG EXIF ---

def _ratio_to_float(r):
    """piexif gives (numerator, denominator); PIL/IFDRational is float-able."""
    try:
        n, d = r
        return float(n) / float(d) if d else 0.0
    except TypeError:
        return float(r)

def _dms_to_decimal(dms, ref):
    d = _ratio_to_float(dms[0])
    m = _ratio_to_float(dms[1])
    s = _ratio_to_float(dms[2])
    val = d + m / 60.0 + s / 3600.0
    if ref in ("S", "W", b"S", b"W"):
        val = -val
    return val

def extract_gps_from_jpg(path):
    """Return (lat, lon, taken_at_iso_or_None) or None if there are no GPS
    tags in the image. Used as a fallback when a folder has photos but no
    flight log file."""
    try:
        exif_dict = piexif.load(path)
    except Exception:
        return None
    gps = exif_dict.get("GPS") or {}
    if not gps or piexif.GPSIFD.GPSLatitude not in gps:
        return None
    try:
        lat = _dms_to_decimal(
            gps[piexif.GPSIFD.GPSLatitude],
            gps.get(piexif.GPSIFD.GPSLatitudeRef, b"N").decode(errors="ignore"),
        )
        lon = _dms_to_decimal(
            gps[piexif.GPSIFD.GPSLongitude],
            gps.get(piexif.GPSIFD.GPSLongitudeRef, b"E").decode(errors="ignore"),
        )
    except Exception:
        return None
    taken_at = None
    dt_raw = exif_dict.get("Exif", {}).get(piexif.ExifIFD.DateTimeOriginal)
    if dt_raw:
        try:
            taken_at = datetime.strptime(
                dt_raw.decode(), "%Y:%m:%d %H:%M:%S"
            ).isoformat()
        except Exception:
            taken_at = None
    return (lat, lon, taken_at)

## 3. Data import

Walks the folder tree the same way v1 did, but with three additions:

* Every photo / video is recorded in `flight_media` with its file path
  and (for JPGs) any extracted EXIF GPS + timestamp.
* If a personal flight has no `.SRT` but its JPGs are geotagged, the
  EXIF coordinates are inserted into `personal_coordinates` instead,
  ordered by photo timestamp. That means the flight will still appear
  on the map.
* Errors on individual files no longer abort the whole import.

In [ ]:
# Helpers for classifying file types and whether to mount Drive.

PHOTO_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".dng"}
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi"}
LOG_EXTS   = {".mrk", ".srt"}

def classify_media(filename):
    ext = os.path.splitext(filename)[1].lower()
    if ext in PHOTO_EXTS: return "photo"
    if ext in VIDEO_EXTS: return "video"
    if ext in LOG_EXTS:   return "log"
    return "other"

def mount_drive_if_needed():
    if IN_COLAB:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
        print("Google Drive mounted.")
    else:
        print("Local mode — skipping Drive mount.")

In [ ]:
def import_folder_data(folder_path, db_path=DB_PATH, verbose=True):
    """Walk a year folder of the form <year>/<month>/<flight>/ and import
    every recognized flight, its coordinate path, and its media inventory."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    stats = {"pro_flights": 0, "per_flights": 0, "media": 0, "coords": 0}

    def log(msg):
        if verbose:
            print(msg)

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Not a directory: {folder_path}")

    for month in os.listdir(folder_path):
        month_path = os.path.join(folder_path, month)
        if not os.path.isdir(month_path):
            continue

        for name in os.listdir(month_path):
            full_path = os.path.join(month_path, name)
            if not os.path.isdir(full_path):
                continue

            try:
                files = os.listdir(full_path)
            except OSError as e:
                log(f"  cannot list {full_path}: {e}")
                continue

            # --- Classify the flight by name ---
            is_pro = name.startswith("DJI_") and "_" in name
            try:
                if is_pro:
                    parts = name.split("_")
                    parsed_date = datetime.strptime(
                        parts[1], "%Y%m%d%H%M"
                    ).strftime("%Y-%m-%d")
                    flight_type = "professional"
                    table = "professional"
                    coord_table = "professional_coordinates"
                    coord_name_col = "professional_name"
                else:
                    # Personal naming: 'mm.ddLocation' (e.g. '10.17Park' or '10.5Lake')
                    if name.count(".") != 1:
                        log(f"  skipping unrecognized folder name: {name}")
                        continue
                    month_part, rest = name.split(".", 1)
                    if len(rest) >= 2 and rest[1].isdigit():
                        day = rest[:2]
                    else:
                        day = rest[:1]
                    year_match = re.search(r"(20\d{2})", folder_path)
                    year = year_match.group(1) if year_match else "0000"
                    parsed_date = datetime.strptime(
                        f"{year}-{month_part}-{day}", "%Y-%m-%d"
                    ).strftime("%Y-%m-%d")
                    flight_type = "personal"
                    table = "personal"
                    coord_table = "personal_coordinates"
                    coord_name_col = "personal_name"
            except Exception as e:
                log(f"  failed to parse '{name}': {e}")
                continue

            # --- Insert / upsert the flight row ---
            cur.execute(
                f"INSERT OR IGNORE INTO {table} (name, date, folder_path) VALUES (?, ?, ?)",
                (name, parsed_date, full_path),
            )
            if cur.rowcount:
                stats[f"{'pro' if is_pro else 'per'}_flights"] += 1
                log(f"  + {flight_type}: {name} ({parsed_date})")

            # --- Refresh coordinates and media for this flight ---
            cur.execute(
                f"DELETE FROM {coord_table} WHERE {coord_name_col} = ?", (name,)
            )
            cur.execute(
                "DELETE FROM flight_media WHERE flight_type = ? AND flight_name = ?",
                (flight_type, name),
            )

            # 1. Inventory all media files first
            jpg_paths_with_exif = []  # for the EXIF-fallback case below
            for fname in files:
                fpath = os.path.join(full_path, fname)
                if not os.path.isfile(fpath):
                    continue
                kind = classify_media(fname)
                lat = lon = taken = None
                if kind == "photo" and fname.lower().endswith((".jpg", ".jpeg")):
                    gps = extract_gps_from_jpg(fpath)
                    if gps:
                        lat, lon, taken = gps
                        jpg_paths_with_exif.append((taken or "", lat, lon))
                cur.execute(
                    """INSERT OR REPLACE INTO flight_media
                       (flight_type, flight_name, filename, file_path,
                        media_kind, lat, lon, taken_at)
                       VALUES (?,?,?,?,?,?,?,?)""",
                    (flight_type, name, fname, fpath, kind, lat, lon, taken),
                )
                stats["media"] += 1

            # 2. Coordinate path: prefer .MRK/.SRT, fall back to JPG EXIF
            log_files = [f for f in files if f.lower().endswith((".mrk", ".srt"))]
            coords_inserted = 0
            if log_files:
                for lf in log_files:
                    lf_path = os.path.join(full_path, lf)
                    if lf.lower().endswith(".mrk"):
                        flight_coords = extract_lat_lon_mrk(lf_path)
                    else:
                        flight_coords = extract_lat_lon_srt(lf_path)
                    for seq, (lat, lon, _) in enumerate(flight_coords):
                        cur.execute(
                            f"""INSERT INTO {coord_table}
                                ({coord_name_col}, lat, lon, sequence)
                                VALUES (?,?,?,?)""",
                            (name, lat, lon, seq),
                        )
                        coords_inserted += 1
            elif jpg_paths_with_exif:
                # No flight log — order EXIF points by capture timestamp.
                jpg_paths_with_exif.sort(key=lambda x: x[0])
                for seq, (_t, lat, lon) in enumerate(jpg_paths_with_exif):
                    cur.execute(
                        f"""INSERT INTO {coord_table}
                            ({coord_name_col}, lat, lon, sequence)
                            VALUES (?,?,?,?)""",
                        (name, lat, lon, seq),
                    )
                    coords_inserted += 1
                log(f"    (no .MRK/.SRT — recovered {coords_inserted} points from JPG EXIF)")
            else:
                log(f"    no coordinate source for {name}")
            stats["coords"] += coords_inserted

    conn.commit()
    conn.close()
    log(f"\nImport complete: {stats}")
    return stats

## 4. Flask web app

The app exposes:

| Route | Purpose |
|---|---|
| `/` | Search form (date range + type + name substring) |
| `/search` | JSON results, with photo / video counts and a link to each flight |
| `/flight/<type>/<name>` | Per-flight detail page — metadata + small map + media gallery |
| `/api/flight/<type>/<name>` | JSON for the detail page |
| `/map` | Combined flight-path map; respects `?date_from=&date_to=` |
| `/api/flights` | Path data for the map; same filter params |
| `/media/<type>/<name>/<file>` | Serves a media file from disk so the gallery can render thumbnails |


In [ ]:
app = Flask(__name__)

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def fetch_flight_summary(conn, flight_type, name):
    table = "professional" if flight_type == "professional" else "personal"
    row = conn.execute(
        f"SELECT name, date, folder_path FROM {table} WHERE name = ?", (name,)
    ).fetchone()
    if not row:
        return None
    counts = conn.execute(
        """SELECT media_kind, COUNT(*) as c FROM flight_media
            WHERE flight_type=? AND flight_name=?
            GROUP BY media_kind""",
        (flight_type, name),
    ).fetchall()
    counts_dict = {r["media_kind"]: r["c"] for r in counts}
    return {
        "type": flight_type,
        "name": row["name"],
        "date": row["date"],
        "folder_path": row["folder_path"],
        "n_photos": counts_dict.get("photo", 0),
        "n_videos": counts_dict.get("video", 0),
        "n_logs":   counts_dict.get("log", 0),
    }

In [ ]:
# ----- HTML templates -----

SEARCH_HTML = """
<!doctype html>
<title>Drone Imagery Search</title>
<style>
  body { font-family: system-ui, Arial, sans-serif; max-width: 900px; margin: 2em auto; padding: 0 1em; color:#222 }
  .form { background:#f5f5f5; padding:1em 1.5em; border-radius:8px; }
  .form .row { margin: .6em 0; display:flex; gap:1em; align-items:center; flex-wrap:wrap }
  label { font-weight:600; min-width:120px; }
  input, select { padding:.4em .5em; border:1px solid #ccc; border-radius:4px; }
  button { background:#2563eb; color:#fff; padding:.6em 1.2em; border:0; border-radius:4px; cursor:pointer; }
  button:hover { background:#1d4ed8; }
  .item { border:1px solid #ddd; padding:.8em 1em; margin:.5em 0; border-radius:6px; }
  .badge { display:inline-block; padding:2px 8px; border-radius:4px; color:#fff; font-size:.85em }
  .professional { background:#2563eb }
  .personal     { background:#16a34a }
  a { color:#2563eb; text-decoration:none } a:hover { text-decoration:underline }
  .muted { color:#666; font-size:.9em }
</style>
<h1>Drone Imagery Search</h1>
<p class="muted">CEEDS Spatial Analysis Lab · CSC230 project</p>
<form class="form" id="f" onsubmit="event.preventDefault(); run();">
  <div class="row"><label>From date</label><input type="date" id="date_from" required></div>
  <div class="row"><label>To date</label>  <input type="date" id="date_to"   required></div>
  <div class="row"><label>Type</label>
    <select id="imagery_type">
      <option value="all">All</option>
      <option value="professional">Professional</option>
      <option value="personal">Personal</option>
    </select>
  </div>
  <div class="row"><label>Name contains</label>
    <input type="text" id="q" placeholder="e.g. DJI_2024 or Park">
  </div>
  <div class="row"><button>Search</button>
    <a id="maplink" href="/map" target="_blank">Open map for these dates →</a>
  </div>
</form>
<div id="out"></div>
<script>
async function run() {
  const df = document.getElementById('date_from').value;
  const dt = document.getElementById('date_to').value;
  const ty = document.getElementById('imagery_type').value;
  const q  = document.getElementById('q').value;
  const params = new URLSearchParams({date_from: df, date_to: dt, imagery_type: ty, q: q});
  document.getElementById('maplink').href = '/map?' + params.toString();
  const out = document.getElementById('out');
  out.innerHTML = '<p>Searching…</p>';
  const r = await fetch('/search?' + params.toString());
  const d = await r.json();
  if (d.error) { out.innerHTML = '<p style="color:#b91c1c">' + d.error + '</p>'; return; }
  if (!d.results.length) { out.innerHTML = '<p>No flights match.</p>'; return; }
  out.innerHTML = '<h2>' + d.results.length + ' flight(s)</h2>' +
    d.results.map(it => `
      <div class="item">
        <div><span class="badge ${it.type}">${it.type}</span>
             <strong><a href="/flight/${it.type}/${encodeURIComponent(it.name)}">${it.name}</a></strong>
        </div>
        <div class="muted">${it.date} · ${it.n_photos} photo(s), ${it.n_videos} video(s)</div>
      </div>
    `).join('');
}
</script>
"""

MAP_HTML = """
<!doctype html>
<title>Flight Map</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<style>
  body { font-family: system-ui, Arial; margin: 1em; }
  #map { height: 78vh; border:1px solid #ccc; }
  .muted { color:#666; font-size:.9em }
</style>
<a href="/">← Back to search</a>
<h1>Drone Flight Paths</h1>
<p class="muted" id="status">Loading…</p>
<div id="map"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const params = new URLSearchParams(window.location.search);
const map = L.map('map').setView([42.3251, -72.6412], 13);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
  { maxZoom: 18, attribution: '© OpenStreetMap' }).addTo(map);

fetch('/api/flights?' + params.toString())
  .then(r => r.json())
  .then(data => {
    const flights = data.flights || {};
    const names = Object.keys(flights);
    document.getElementById('status').textContent =
      names.length ? `Showing ${names.length} flight(s)` : 'No flights match the filter';
    const all = [];
    names.forEach(name => {
      const f = flights[name];
      const color = f.type === 'professional' ? '#2563eb' : '#16a34a';
      const line = L.polyline(f.coords, {color, weight:3}).addTo(map);
      const link = `<a href="/flight/${f.type}/${encodeURIComponent(name)}" target="_blank">open details</a>`;
      line.bindPopup(`<b>${name}</b><br>${f.date} · ${f.type}<br>${link}`);
      f.coords.forEach(c => all.push(c));
    });
    if (all.length) map.fitBounds(L.latLngBounds(all).pad(0.1));
  })
  .catch(e => { document.getElementById('status').textContent = 'Error: ' + e.message; });
</script>
"""

FLIGHT_HTML = """
<!doctype html>
<title>Flight {{ name }}</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<style>
  body { font-family: system-ui, Arial; max-width:1100px; margin:1em auto; padding:0 1em; }
  #map { height: 360px; border:1px solid #ccc; margin: 1em 0; }
  .gallery { display:grid; grid-template-columns: repeat(auto-fill, minmax(160px,1fr)); gap:.5em; }
  .thumb { border:1px solid #ddd; border-radius:6px; overflow:hidden; background:#fafafa; }
  .thumb img, .thumb video { width:100%; height:120px; object-fit:cover; display:block; }
  .thumb .cap { padding:.3em .5em; font-size:.8em; color:#444; word-break:break-all; }
  .badge { display:inline-block; padding:2px 8px; border-radius:4px; color:#fff; font-size:.85em }
  .professional { background:#2563eb } .personal { background:#16a34a }
  .muted { color:#666 }
</style>
<a href="/">← Back to search</a>
<h1>{{ name }}</h1>
<p><span class="badge {{ type }}">{{ type }}</span>
   <strong>{{ date }}</strong> ·
   {{ n_photos }} photos, {{ n_videos }} videos, {{ n_logs }} log file(s)</p>
<p class="muted">Folder: <code>{{ folder_path }}</code></p>
<div id="map"></div>
<h2>Media</h2>
<div class="gallery" id="gallery">Loading…</div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
const map = L.map('map').setView([42.3251, -72.6412], 14);
L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
  { maxZoom: 18, attribution: '© OpenStreetMap' }).addTo(map);

const flightType = {{ type|tojson }};
const flightName = {{ name|tojson }};

fetch(`/api/flight/${flightType}/${encodeURIComponent(flightName)}`)
  .then(r => r.json())
  .then(d => {
    if (d.coords && d.coords.length) {
      const line = L.polyline(d.coords, {color:'#2563eb', weight:3}).addTo(map);
      map.fitBounds(line.getBounds().pad(0.2));
    }
    const g = document.getElementById('gallery');
    if (!d.media || !d.media.length) { g.textContent = 'No media files.'; return; }
    g.innerHTML = d.media.map(m => {
      const url = `/media/${flightType}/${encodeURIComponent(flightName)}/${encodeURIComponent(m.filename)}`;
      if (m.media_kind === 'photo') {
        return `<div class="thumb"><a href="${url}" target="_blank"><img loading="lazy" src="${url}"></a><div class="cap">${m.filename}</div></div>`;
      }
      if (m.media_kind === 'video') {
        return `<div class="thumb"><video controls preload="metadata" src="${url}"></video><div class="cap">${m.filename}</div></div>`;
      }
      return `<div class="thumb"><div style="height:120px;display:flex;align-items:center;justify-content:center;color:#888">${m.media_kind}</div><div class="cap">${m.filename}</div></div>`;
    }).join('');
  });
</script>
"""


In [ ]:
# ----- Routes -----

@app.route("/")
def index():
    return render_template_string(SEARCH_HTML)

@app.route("/search")
def search():
    df = request.args.get("date_from")
    dt = request.args.get("date_to")
    ty = request.args.get("imagery_type", "all")
    q  = (request.args.get("q") or "").strip()
    if not df or not dt:
        return jsonify({"error": "Both dates are required"}), 400
    try:
        datetime.strptime(df, "%Y-%m-%d")
        datetime.strptime(dt, "%Y-%m-%d")
    except ValueError:
        return jsonify({"error": "Invalid date format (use YYYY-MM-DD)"}), 400

    conn = get_db()
    results = []
    try:
        like = f"%{q}%" if q else "%"
        if ty in ("all", "professional"):
            for r in conn.execute("""
                SELECT p.name, p.date, p.folder_path,
                       COALESCE(SUM(fm.media_kind='photo'), 0) AS n_photos,
                       COALESCE(SUM(fm.media_kind='video'), 0) AS n_videos
                FROM professional p
                LEFT JOIN flight_media fm
                       ON fm.flight_type='professional' AND fm.flight_name=p.name
                WHERE p.date BETWEEN ? AND ? AND p.name LIKE ?
                GROUP BY p.name, p.date, p.folder_path
                ORDER BY p.date
            """, (df, dt, like)):
                results.append({**dict(r), "type": "professional"})
        if ty in ("all", "personal"):
            for r in conn.execute("""
                SELECT p.name, p.date, p.folder_path,
                       COALESCE(SUM(fm.media_kind='photo'), 0) AS n_photos,
                       COALESCE(SUM(fm.media_kind='video'), 0) AS n_videos
                FROM personal p
                LEFT JOIN flight_media fm
                       ON fm.flight_type='personal' AND fm.flight_name=p.name
                WHERE p.date BETWEEN ? AND ? AND p.name LIKE ?
                GROUP BY p.name, p.date, p.folder_path
                ORDER BY p.date
            """, (df, dt, like)):
                results.append({**dict(r), "type": "personal"})
        return jsonify({"results": results})
    finally:
        conn.close()

@app.route("/api/flights")
def api_flights():
    df = request.args.get("date_from")
    dt = request.args.get("date_to")
    ty = request.args.get("imagery_type", "all")
    q  = (request.args.get("q") or "").strip()

    where_pro, where_per, params_pro, params_per = "1=1", "1=1", [], []
    if df and dt:
        where_pro += " AND p.date BETWEEN ? AND ?";  params_pro += [df, dt]
        where_per += " AND p.date BETWEEN ? AND ?";  params_per += [df, dt]
    if q:
        where_pro += " AND p.name LIKE ?"; params_pro.append(f"%{q}%")
        where_per += " AND p.name LIKE ?"; params_per.append(f"%{q}%")

    flights = {}
    conn = get_db()
    try:
        if ty in ("all", "professional"):
            for r in conn.execute(f"""
                SELECT p.name, p.date, pc.lat, pc.lon, pc.sequence
                FROM professional p
                JOIN professional_coordinates pc ON pc.professional_name = p.name
                WHERE {where_pro}
                ORDER BY p.name, pc.sequence
            """, params_pro):
                f = flights.setdefault(r["name"], {"type": "professional",
                                                  "date": r["date"],
                                                  "coords": []})
                f["coords"].append([r["lat"], r["lon"]])
        if ty in ("all", "personal"):
            for r in conn.execute(f"""
                SELECT p.name, p.date, pc.lat, pc.lon, pc.sequence
                FROM personal p
                JOIN personal_coordinates pc ON pc.personal_name = p.name
                WHERE {where_per}
                ORDER BY p.name, pc.sequence
            """, params_per):
                f = flights.setdefault(r["name"], {"type": "personal",
                                                  "date": r["date"],
                                                  "coords": []})
                f["coords"].append([r["lat"], r["lon"]])
        return jsonify({"flights": flights})
    finally:
        conn.close()

@app.route("/map")
def map_page():
    return render_template_string(MAP_HTML)

@app.route("/flight/<flight_type>/<path:name>")
def flight_page(flight_type, name):
    if flight_type not in ("professional", "personal"):
        abort(404)
    conn = get_db()
    try:
        summary = fetch_flight_summary(conn, flight_type, name)
    finally:
        conn.close()
    if not summary:
        abort(404)
    return render_template_string(FLIGHT_HTML, **summary)

@app.route("/api/flight/<flight_type>/<path:name>")
def api_flight(flight_type, name):
    if flight_type not in ("professional", "personal"):
        abort(404)
    conn = get_db()
    try:
        summary = fetch_flight_summary(conn, flight_type, name)
        if not summary:
            abort(404)
        coord_table = (
            "professional_coordinates" if flight_type == "professional"
            else "personal_coordinates"
        )
        coord_col = (
            "professional_name" if flight_type == "professional"
            else "personal_name"
        )
        coords = [[r["lat"], r["lon"]] for r in conn.execute(
            f"SELECT lat, lon FROM {coord_table} WHERE {coord_col}=? ORDER BY sequence",
            (name,),
        )]
        media = [dict(r) for r in conn.execute(
            """SELECT filename, media_kind, lat, lon, taken_at
               FROM flight_media
               WHERE flight_type=? AND flight_name=?
               ORDER BY taken_at, filename""",
            (flight_type, name),
        )]
        return jsonify({**summary, "coords": coords, "media": media})
    finally:
        conn.close()

@app.route("/media/<flight_type>/<flight_name>/<path:filename>")
def serve_media(flight_type, flight_name, filename):
    """Serve a single media file by looking it up in flight_media. Looking
    it up (rather than concatenating paths) prevents directory traversal."""
    if flight_type not in ("professional", "personal"):
        abort(404)
    conn = get_db()
    try:
        row = conn.execute(
            """SELECT file_path FROM flight_media
               WHERE flight_type=? AND flight_name=? AND filename=?""",
            (flight_type, flight_name, filename),
        ).fetchone()
    finally:
        conn.close()
    if not row or not os.path.isfile(row["file_path"]):
        abort(404)
    mime, _ = mimetypes.guess_type(row["file_path"])
    return send_file(row["file_path"], mimetype=mime or "application/octet-stream")


## 5. Run the server

In Colab, this uses the `google.colab.output.eval_js` proxy trick to embed
the Flask app in an iframe. Locally, it just prints the URL — open it in
your browser.

In [ ]:
def find_free_port(start=9000):
    p = start
    while True:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.bind(("0.0.0.0", p))
                return p
        except OSError:
            p += 1

def run_server():
    port = find_free_port()
    th = threading.Thread(
        target=lambda: app.run(host="0.0.0.0", port=port,
                                debug=False, use_reloader=False),
        daemon=True,
    )
    th.start()
    time.sleep(1.5)

    if IN_COLAB:
        proxy_url = output.eval_js(
            f"google.colab.kernel.proxyPort({port})"
        )
        display(HTML(f"""
            <h2>Drone Imagery Search</h2>
            <p>
              <a href="{proxy_url}" target="_blank">Open search</a> ·
              <a href="{proxy_url}map" target="_blank">Open map</a>
            </p>
            <iframe src="{proxy_url}" width="100%" height="600"></iframe>
            <p>Running on port {port}.</p>
        """))
    else:
        print(f"Server running at http://localhost:{port}/")
        print(f"  Search:    http://localhost:{port}/")
        print(f"  Map:       http://localhost:{port}/map")
    return port

# Uncomment to run:
# run_server()

## 6. Inspect the database

Quick sanity check — counts per table and a sample of each.

In [ ]:
def show_database_contents(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    print("Tables:")
    print(pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
    ).to_string(index=False))

    for t in ("professional", "personal",
              "professional_coordinates", "personal_coordinates",
              "flight_media"):
        n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0]["n"]
        print(f"\n{t}: {n} row(s)")
        if n:
            display(pd.read_sql_query(f"SELECT * FROM {t} LIMIT 5", conn))
    conn.close()

# show_database_contents()

## 7. Local test mode — synthetic data

This section builds a fake folder tree under `/tmp/drones_synth/2024/`
that exercises every code path:

* `10/DJI_202410171432_001` — professional flight with a `.MRK` file
* `10/10.17Park` — personal flight with a `.SRT` file
* `10/10.18Trail` — personal flight with **only geotagged JPGs** (this is
  the case the proposal flagged: location must be inferred from EXIF)

Run all three cells in order to see the full pipeline against synthetic
data, with no Google Drive needed.

In [ ]:
import shutil
from fractions import Fraction

SYNTH_ROOT = "/tmp/drones_synth/2024"

def _deg_to_dms_rational(deg):
    deg = abs(deg)
    d = int(deg)
    m_f = (deg - d) * 60
    m = int(m_f)
    s = round((m_f - m) * 60 * 100)
    return ((d, 1), (m, 1), (s, 100))

def _write_geotagged_jpg(path, lat, lon, dt):
    Image.new("RGB", (40, 40), color="navy").save(path, "jpeg")
    exif = {
        "0th": {piexif.ImageIFD.DateTime: dt.strftime("%Y:%m:%d %H:%M:%S").encode()},
        "Exif": {piexif.ExifIFD.DateTimeOriginal: dt.strftime("%Y:%m:%d %H:%M:%S").encode()},
        "GPS": {
            piexif.GPSIFD.GPSLatitudeRef:  b"N" if lat >= 0 else b"S",
            piexif.GPSIFD.GPSLatitude:     _deg_to_dms_rational(lat),
            piexif.GPSIFD.GPSLongitudeRef: b"E" if lon >= 0 else b"W",
            piexif.GPSIFD.GPSLongitude:    _deg_to_dms_rational(lon),
        },
        "1st": {}, "thumbnail": None,
    }
    piexif.insert(piexif.dump(exif), path)

def build_synthetic_data():
    if os.path.exists(SYNTH_ROOT):
        shutil.rmtree(SYNTH_ROOT)
    oct_dir = os.path.join(SYNTH_ROOT, "10")
    os.makedirs(oct_dir)

    # --- Professional flight with an .MRK file ---
    pro = os.path.join(oct_dir, "DJI_202410171432_001")
    os.makedirs(pro)
    with open(os.path.join(pro, "Timestamp.MRK"), "w") as f:
        path = [(42.3175 + i*0.0002, -72.6395 + i*0.0001) for i in range(8)]
        for i, (lat, lon) in enumerate(path):
            f.write(f"{i}\t1\t{lat},Lat,{lon},Lon,80.0,Ellh\n")
    for i in range(3):
        _write_geotagged_jpg(
            os.path.join(pro, f"DJI_{i:03d}.JPG"),
            42.3175 + i*0.0002, -72.6395 + i*0.0001,
            datetime(2024, 10, 17, 14, 32, i),
        )

    # --- Personal flight with an .SRT video log ---
    per_srt = os.path.join(oct_dir, "10.17Park")
    os.makedirs(per_srt)
    with open(os.path.join(per_srt, "DJI_video.SRT"), "w") as f:
        for i in range(6):
            f.write(f"{i+1}\n00:00:0{i},000 --> 00:00:0{i+1},000\n"
                    f"<font size=\"28\">[latitude: {42.319 + i*0.0001}] "
                    f"[longitude: {-72.638 + i*0.0001}] [altitude: 80]</font>\n\n")
    # also drop in a fake mp4 (empty file — just for media-listing demo)
    open(os.path.join(per_srt, "DJI_video.mp4"), "wb").close()

    # --- Personal flight with ONLY geotagged JPGs (the EXIF-fallback case) ---
    per_exif = os.path.join(oct_dir, "10.18Trail")
    os.makedirs(per_exif)
    for i in range(5):
        _write_geotagged_jpg(
            os.path.join(per_exif, f"IMG_{i:03d}.JPG"),
            42.321 + i*0.0003, -72.640 + i*0.00015,
            datetime(2024, 10, 18, 11, i, 0),
        )

    print(f"Built synthetic data at {SYNTH_ROOT}")
    for d in [pro, per_srt, per_exif]:
        print(f"  {os.path.relpath(d, SYNTH_ROOT)}: {sorted(os.listdir(d))}")

# Run it
build_synthetic_data()

In [ ]:
# Reset DB then import the synthetic folder. Comment out the os.remove call
# if you want to layer synthetic data on top of an existing import.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
setup_database()
import_folder_data(SYNTH_ROOT)
show_database_contents()

In [ ]:
# Start the server against synthetic data. Open the URL it prints (or use
# the embedded iframe in Colab) and try:
#   - search 2024-10-01 to 2024-10-31, type=All
#   - check that 10.18Trail has 5 photos and a flight path on the map
#     (this is the EXIF-only case)
#   - click any flight name to see its media gallery
run_server()